In [0]:
from pyspark.sql import functions as F

episodios = spark.table("mvp_engdados.silver.episodes")
imdb = spark.table("mvp_engdados.silver.imdb")

In [0]:
display(
    episodios.groupBy("series_id")
    .count()
    .orderBy("series_id")
)

display(
    imdb.groupBy("series_id")
    .count()
    .orderBy("series_id")
)

display(
    episodios
    .filter(F.col("original_air_date").isNull())
    .select("series_id", "title", "original_air_date_raw")
)

display(
    imdb
    .filter(F.col("original_air_date").isNull())
    .select("series_id", "title", "original_air_date_raw")
)

series_id,count
big_bang_theory,279
breaking_bad,62
brooklyn_99,153
friends,236
game_of_thrones,73
how_i_met_your_mother,208
lost,121
the_office,201
the_walking_dead,177


series_id,count
big_bang_theory,280
breaking_bad,62
brooklyn_99,153
friends,235
game_of_thrones,73
how_i_met_your_mother,208
lost,119
the_office,188
the_walking_dead,177


series_id,title,original_air_date_raw


series_id,title,original_air_date_raw


In [0]:
display(
    episodios
    .groupBy("series_id", "season", "episode_num_in_season")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("series_id", "season", "episode_num_in_season")
)

display(
    imdb
    .groupBy("series_id", "season", "episode_num")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("series_id", "season", "episode_num")
)

series_id,season,episode_num_in_season,count


series_id,season,episode_num,count


In [0]:
display(
    imdb.filter(
        (F.col("imdb_rating") < 0) |
        (F.col("imdb_rating") > 10) |
        (F.col("total_votes") < 0)
    )
)

display(
    episodios.filter(F.col("us_viewers") < 0)
)

season,episode_num,title,original_air_date_raw,imdb_rating,total_votes,desc,series_id,_source_file,_ingested_at,original_air_date


season,episode_num_in_season,episode_num_overall,title,directed_by,written_by,original_air_date_raw,prod_code,us_viewers,series_id,_source_file,_ingested_at,featured_character,episode_num_in_season_raw,original_air_date


In [0]:
e = episodios.alias("e")
i = imdb.alias("i")

condicao = (
    (F.col("e.series_id") == F.col("i.series_id")) &
    (F.col("e.season") == F.col("i.season")) &
    (F.col("e.episode_num_in_season") == F.col("i.episode_num"))
)

reconciliacao = (
    e.join(i, condicao, "full_outer")
    .select(
        F.coalesce(F.col("e.series_id"), F.col("i.series_id")).alias("series_id"),
        F.coalesce(F.col("e.season"), F.col("i.season")).alias("season"),
        F.coalesce(
            F.col("e.episode_num_in_season"),
            F.col("i.episode_num")
        ).alias("episode_num_candidate"),
        F.col("e.title").alias("episode_title"),
        F.col("i.title").alias("imdb_title"),
        F.col("e.original_air_date").alias("episode_date"),
        F.col("i.original_air_date").alias("imdb_date"),
        F.col("e.episode_num_overall").alias("episode_num_overall"),
        F.col("i.imdb_rating").alias("imdb_rating"),
        F.col("i.total_votes").alias("total_votes")
    )
    .withColumn(
        "title_match",
        F.lower(F.trim("episode_title")) == F.lower(F.trim("imdb_title"))
    )
    .withColumn("date_match", F.col("episode_date") == F.col("imdb_date"))
    .withColumn(
        "match_status",
        F.when(F.col("episode_title").isNull(), "somente_imdb")
         .when(F.col("imdb_title").isNull(), "somente_episodes")
         .when(F.col("title_match") & F.col("date_match"), "confirmado")
         .when(F.col("title_match"), "titulo_igual_data_diferente")
         .when(F.col("date_match"), "data_igual_titulo_diferente")
         .otherwise("titulo_e_data_diferentes")
    )
)

In [0]:
display(
    reconciliacao
    .groupBy("series_id", "match_status")
    .count()
    .orderBy("series_id", "match_status")
)

series_id,match_status,count
big_bang_theory,confirmado,262
big_bang_theory,somente_imdb,1
big_bang_theory,titulo_e_data_diferentes,17
breaking_bad,confirmado,62
brooklyn_99,confirmado,153
friends,confirmado,235
friends,somente_episodes,1
game_of_thrones,confirmado,73
how_i_met_your_mother,confirmado,208
lost,confirmado,119


In [0]:
display(
    reconciliacao
    .filter(F.col("match_status") != "confirmado")
    .orderBy("series_id", "season", "episode_num_candidate")
)

series_id,season,episode_num_candidate,episode_title,imdb_title,episode_date,imdb_date,episode_num_overall,imdb_rating,total_votes,title_match,date_match,match_status
big_bang_theory,1,1,Pilot,Unaired Pilot,2007-09-24,2006-05-01,1,6.7,2048,false,false,titulo_e_data_diferentes
big_bang_theory,1,2,The Big Bran Hypothesis,Pilot,2007-10-01,2007-09-24,2,8.2,6135,false,false,titulo_e_data_diferentes
big_bang_theory,1,3,The Fuzzy Boots Corollary,The Big Bran Hypothesis,2007-10-08,2007-10-01,3,8.3,4924,false,false,titulo_e_data_diferentes
big_bang_theory,1,4,The Luminous Fish Effect,The Fuzzy Boots Corollary,2007-10-15,2007-10-08,4,7.7,4392,false,false,titulo_e_data_diferentes
big_bang_theory,1,5,The Hamburger Postulate,The Luminous Fish Effect,2007-10-22,2007-10-15,5,8.1,4434,false,false,titulo_e_data_diferentes
big_bang_theory,1,6,The Middle Earth Paradigm,The Hamburger Postulate,2007-10-29,2007-10-22,6,7.9,4227,false,false,titulo_e_data_diferentes
big_bang_theory,1,7,The Dumpling Paradox,The Middle Earth Paradigm,2007-11-05,2007-10-29,7,8.4,4402,false,false,titulo_e_data_diferentes
big_bang_theory,1,8,The Grasshopper Experiment,The Dumpling Paradox,2007-11-12,2007-11-05,8,8.1,4143,false,false,titulo_e_data_diferentes
big_bang_theory,1,9,The Cooper-Hofstadter Polarization,The Grasshopper Experiment,2008-03-17,2007-11-12,9,8.2,4172,false,false,titulo_e_data_diferentes
big_bang_theory,1,10,The Loobenfeld Decay,The Cooper-Hofstadter Polarization,2008-03-24,2008-03-17,10,8.0,4024,false,false,titulo_e_data_diferentes


In [0]:
chaves = ["series_id", "_title_key", "_airdate"]

e = (
    episodios
    .filter(
        F.col("title").isNotNull() &
        F.col("original_air_date").isNotNull()
    )
    .withColumn(
        "_title_key",
        F.lower(F.regexp_replace(F.trim("title"), r"\s+", " "))
    )
    .withColumn("_airdate", F.col("original_air_date"))
)

i = (
    imdb
    .filter(
        F.col("title").isNotNull() &
        F.col("original_air_date").isNotNull()
    )
    .withColumn(
        "_title_key",
        F.lower(F.regexp_replace(F.trim("title"), r"\s+", " "))
    )
    .withColumn("_airdate", F.col("original_air_date"))
)

e_unicos = (
    e.join(
        e.groupBy(*chaves)
         .count()
         .filter(F.col("count") == 1)
         .select(*chaves),
        on=chaves,
        how="inner"
    )
)

i_unicos = (
    i.join(
        i.groupBy(*chaves)
         .count()
         .filter(F.col("count") == 1)
         .select(*chaves),
        on=chaves,
        how="inner"
    )
)

pares_seguros = (
    e_unicos.alias("e")
    .join(i_unicos.alias("i"), on=chaves, how="inner")
    .select(
        F.col("e.series_id").alias("series_id"),
        F.col("e.season").alias("episode_season"),
        F.col("e.episode_num_in_season").alias("episode_number"),
        F.col("i.season").alias("imdb_season"),
        F.col("i.episode_num").alias("imdb_episode_number"),
        F.col("e.title").alias("episode_title"),
        F.col("e.original_air_date").alias("air_date"),
        F.col("e.us_viewers").alias("us_viewers"),
        F.col("i.imdb_rating").alias("imdb_rating"),
        F.col("i.total_votes").alias("total_votes")
    )
)

In [0]:
display(
    pares_seguros
    .groupBy("series_id")
    .count()
    .orderBy("series_id")
)

series_id,count
big_bang_theory,279
breaking_bad,62
brooklyn_99,153
friends,234
game_of_thrones,73
how_i_met_your_mother,208
lost,117
the_office,175
the_walking_dead,177


In [0]:
display(
    pares_seguros
    .filter(
        (F.col("episode_season") != F.col("imdb_season")) |
        (F.col("episode_number") != F.col("imdb_episode_number"))
    )
    .orderBy("series_id", "episode_season", "episode_number")
)

series_id,episode_season,episode_number,imdb_season,imdb_episode_number,episode_title,air_date,us_viewers,imdb_rating,total_votes
big_bang_theory,1,1,1,2,Pilot,2007-09-24,9520000.0,8.2,6135
big_bang_theory,1,2,1,3,The Big Bran Hypothesis,2007-10-01,8580000.0,8.3,4924
big_bang_theory,1,3,1,4,The Fuzzy Boots Corollary,2007-10-08,8359999.999999999,7.7,4392
big_bang_theory,1,4,1,5,The Luminous Fish Effect,2007-10-15,8150000.0,8.1,4434
big_bang_theory,1,5,1,6,The Hamburger Postulate,2007-10-22,8810000.0,7.9,4227
big_bang_theory,1,6,1,7,The Middle Earth Paradigm,2007-10-29,8920000.0,8.4,4402
big_bang_theory,1,7,1,8,The Dumpling Paradox,2007-11-05,9680000.0,8.1,4143
big_bang_theory,1,8,1,9,The Grasshopper Experiment,2007-11-12,9320000.0,8.2,4172
big_bang_theory,1,9,1,10,The Cooper-Hofstadter Polarization,2008-03-17,9110000.0,8.0,4024
big_bang_theory,1,10,1,11,The Loobenfeld Decay,2008-03-24,8630000.0,8.0,4119


In [0]:
(
    pares_seguros
    .withColumn(
        "match_method",
        F.lit("series_id + título normalizado + data; chave única nas duas fontes")
    )
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.silver.episodes_imdb_matched")
)

In [0]:
display(
    spark.table("mvp_engdados.silver.episodes_imdb_matched")
    .groupBy("series_id")
    .count()
    .orderBy("series_id")
)

series_id,count
big_bang_theory,279
breaking_bad,62
brooklyn_99,153
friends,234
game_of_thrones,73
how_i_met_your_mother,208
lost,117
the_office,175
the_walking_dead,177
